# 06. SWE mode

Финальный слой: отдельный граф для issue-resolution.
Более строгие guardrails (interrupt на write/edit), только skill `swe-resolution`.

Генерируется `agents/step_06_swe.py` с двумя графами: `agent` и `swe_agent`.

```bash
uv run langgraph dev --config langgraph.steps.json
```

В Deep Agents UI используй Assistant ID: `openclaw` или `openclaw_swe`.

## Визуальная рамка

![SWE mode loop](visuals/06_swe_mode_loop.svg)

Этот шаг делает отдельный `openclaw_swe` assistant: reproduce → localize → patch → regression coverage → tests, с более строгими write/edit guardrails.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend, LocalShellBackend
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'pyproject.toml').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

from connectors import CONNECTOR_TOOLS

In [ ]:
SUBAGENTS = [
    {"name": "repo-researcher", "description": "Map repository structure, APIs, tests, and likely change locations.", "system_prompt": "You research codebases. Inspect files, identify relevant modules, and return concise findings with file paths and rationale."},
    {"name": "reviewer", "description": "Review proposed patches for bugs, missing tests, and regressions.", "system_prompt": "You are a code reviewer. Focus on correctness, edge cases, tests, security, and maintainability. Be specific and concise."},
]

In [ ]:
DEFAULT_MODEL = 'openrouter:tencent/hy3:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)


def workspace_root() -> Path:
    return Path(os.getenv('OPENCLAW_WORKSPACE', '.')).expanduser().resolve()


def backend():
    root_dir = workspace_root()
    if os.getenv('OPENCLAW_ENABLE_LOCAL_SHELL') != '1':
        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)
    return LocalShellBackend(
        root_dir=root_dir, virtual_mode=True,
        inherit_env=True, timeout=120, max_output_bytes=80_000,
    )


BASE_PROMPT = """\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.
"""

SWE_PROMPT = BASE_PROMPT + """\

You are running in SWE mode. Optimize for issue resolution:
- reproduce or characterize the failure first;
- localize the root cause before patching;
- add regression coverage where practical;
- run the narrowest useful verification before broad checks;
- keep a clear chain from issue to patch to test.
"""


agent = create_deep_agent(
    model=model_name(),
    tools=CONNECTOR_TOOLS,
    system_prompt=BASE_PROMPT,
    subagents=SUBAGENTS,
    skills=["/skills/swe-resolution", "/skills/product-research"],
    memory=["/AGENTS.md"],
    backend=backend(),
    interrupt_on={"execute": True},
)

swe_agent = create_deep_agent(
    model=model_name(),
    tools=CONNECTOR_TOOLS,
    system_prompt=SWE_PROMPT,
    subagents=SUBAGENTS,
    skills=["/skills/swe-resolution"],
    memory=["/AGENTS.md"],
    backend=backend(),
    interrupt_on={"execute": True, "write_file": True, "edit_file": True},
)

print(type(agent).__name__, type(swe_agent).__name__)

### Сгенерировать entrypoint

In [ ]:
import json

AGENTS_DIR = REPO_ROOT / "agents"
AGENTS_DIR.mkdir(exist_ok=True)

agent_code = '''"""Step 06: general agent + SWE-mode agent."""

from __future__ import annotations

import os
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend, LocalShellBackend
from dotenv import load_dotenv

from connectors import CONNECTOR_TOOLS

load_dotenv()

DEFAULT_MODEL = "openrouter:tencent/hy3:free"


def _workspace_root() -> Path:
    return Path(os.getenv("OPENCLAW_WORKSPACE", ".")).expanduser().resolve()


def _backend():
    root_dir = _workspace_root()
    if os.getenv("OPENCLAW_ENABLE_LOCAL_SHELL") != "1":
        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)
    return LocalShellBackend(
        root_dir=root_dir, virtual_mode=True,
        inherit_env=True, timeout=120, max_output_bytes=80_000,
    )


SUBAGENTS = [
    {"name": "repo-researcher", "description": "Map repository structure, APIs, tests, and likely change locations.", "system_prompt": "You research codebases. Inspect files, identify relevant modules, and return concise findings with file paths and rationale."},
    {"name": "reviewer", "description": "Review proposed patches for bugs, missing tests, and regressions.", "system_prompt": "You are a code reviewer. Focus on correctness, edge cases, tests, security, and maintainability. Be specific and concise."},
]

BASE_PROMPT = """\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.
"""

SWE_PROMPT = BASE_PROMPT + """\

You are running in SWE mode. Optimize for issue resolution:
- reproduce or characterize the failure first;
- localize the root cause before patching;
- add regression coverage where practical;
- run the narrowest useful verification before broad checks;
- keep a clear chain from issue to patch to test.
"""


agent = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL),
    tools=CONNECTOR_TOOLS,
    system_prompt=BASE_PROMPT,
    subagents=SUBAGENTS,
    skills=["/skills/swe-resolution", "/skills/product-research"],
    memory=["/AGENTS.md"],
    backend=_backend(),
    interrupt_on={"execute": True},
)

swe_agent = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL),
    tools=CONNECTOR_TOOLS,
    system_prompt=SWE_PROMPT,
    subagents=SUBAGENTS,
    skills=["/skills/swe-resolution"],
    memory=["/AGENTS.md"],
    backend=_backend(),
    interrupt_on={"execute": True, "write_file": True, "edit_file": True},
)
'''

step_file = AGENTS_DIR / "step_06_swe.py"
step_file.write_text(agent_code)

steps_config = {
    "dependencies": ["."],
    "graphs": {
        "openclaw": "./agents/step_06_swe.py:agent",
        "openclaw_swe": "./agents/step_06_swe.py:swe_agent",
    },
    "env": ".env",
}
config_path = REPO_ROOT / "langgraph.steps.json"
config_path.write_text(json.dumps(steps_config, indent=2) + "\n")

print(f"Wrote {step_file}")
print(f"Wrote {config_path}")
print()
print("Restart:")
print("  uv run langgraph dev --config langgraph.steps.json")

### Проверочный prompt

После перезапуска выбери в Deep Agents UI Assistant ID `openclaw_swe` и попробуй:

> "Inspect the project shape test and explain what behavior it protects."